# 002. Meet the air quality data agent: scope, sources and the first ten questions

Code from post 002 of the 365 days of Microsoft Fabric series, in the order the post runs it. The post explains each cell. The post publishes on t-sql.dk on Tuesday 10 November 2026.

*Status: runnable design. The code below has not been run in my workspace yet. It needs the semantic model from post 001. The data agent SDK is in preview, and its tenant and capacity requirements will be checked again in the week before publication. The agent's first scored run is post 016.*

In [ ]:
%pip install semantic-link-labs fabric-data-agent-sdk

In [ ]:
from sempy_labs.tom import connect_semantic_model

MODEL_NAME = "OpenAQ air quality"

DESCRIPTIONS = {
    ("fact_measurement", None): "One hourly reading from one OpenAQ sensor. Several sensors can "
        "measure the same pollutant at one location.",
    ("fact_measurement", "Value"): "The hourly reading in the unit given by dim_parameter[Unit]. "
        "Not a daily mean.",
    ("fact_measurement", "MeasurementUtc"): "When the hour ended, in UTC.",
    ("fact_measurement", "MeasurementLocal"): "When the hour ended, in the station's local time "
        "with its UTC offset, as published by OpenAQ.",
    ("fact_measurement", "SensorId"): "OpenAQ sensor ID. Use it to tell sensors apart. Never add "
        "readings across sensors.",
    ("dim_location", None): "An OpenAQ monitoring station. One row per station.",
    ("dim_location", "ProviderName"): "The organization that shares the data with OpenAQ. Not "
        "necessarily the station owner.",
    ("dim_location", "OwnerName"): "The organization responsible for the station, as recorded by OpenAQ.",
    ("dim_parameter", None): "A pollutant together with its unit. PM2.5 has ParameterName pm25.",
    ("dim_date", None): "Calendar of local reporting dates. A reading belongs to the local date on "
        "which its hour started.",
    ("ref_guideline", None): "WHO 2021 guideline values. They apply to averages over the stated "
        "period, never to single hourly readings. Not related to any other table.",
}

with connect_semantic_model(dataset=MODEL_NAME, readonly=False) as tom:
    for (table, column), text in DESCRIPTIONS.items():
        target = tom.model.Tables[table]
        if column:
            target = target.Columns[column]
        target.Description = text

In [ ]:
import sempy.fabric as fabric
from sempy_labs import resolve_dataset_id
from fabric.dataagent.client import create_data_agent

AGENT_INSTRUCTIONS = """
You answer questions about hourly air-quality readings from OpenAQ in the semantic model
"OpenAQ air quality".
- Daily questions use dim_date, which is the station's local date. Say so in the answer.
- A daily mean here is the average of the hours present. Always report how many hours it uses.
- Several sensors can measure the same pollutant at one location. Do not add their readings.
- Compare with ref_guideline only for daily means against the 24-hour value. Never compare an
  hourly reading with it, and never call a value above it an exceedance.
- "Latest" means the latest reading loaded into the model, not the live OpenAQ feed.
- The data contains no causes and no forecasts. If asked why values changed or what they will
  be, say that the data cannot answer it.
- If the model does not contain what a question needs, say so instead of estimating.
"""

workspace_id = fabric.get_workspace_id()
agent = create_data_agent(data_agent_name="Air quality agent", workspace_id=workspace_id)
agent.update_settings(ai_instructions=AGENT_INSTRUCTIONS)
agent.add_staging_datasource(
    artifact_name_or_id=str(resolve_dataset_id(MODEL_NAME)),
    workspace_id_or_name=workspace_id,
)
agent.publish_staging(description="Week 1: one semantic model, ten-question scope")

In [ ]:
REFERENCE_QUERIES = {
    1: """
EVALUATE
SUMMARIZECOLUMNS (
    dim_location[LocationName],
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    DATESBETWEEN ( dim_date[Date], DATE ( 2026, 9, 1 ), DATE ( 2026, 9, 30 ) ),
    "PM2.5 readings", COUNTROWS ( fact_measurement )
)""",
    2: """
DEFINE
    VAR Berlin = TREATAS ( { 2993 }, dim_location[LocationId] )
    VAR Pm25 = TREATAS ( { "pm25" }, dim_parameter[ParameterName] )
EVALUATE
GENERATE (
    CALCULATETABLE ( VALUES ( fact_measurement[SensorId] ), Berlin, Pm25 ),
    CALCULATETABLE (
        TOPN (
            1,
            SELECTCOLUMNS (
                fact_measurement,
                "Hour ended (UTC)", fact_measurement[MeasurementUtc],
                "Value", fact_measurement[Value]
            ),
            [Hour ended (UTC)], DESC
        ),
        Berlin, Pm25
    )
)""",
    3: """
EVALUATE
SUMMARIZECOLUMNS (
    dim_date[Date],
    TREATAS ( { 2993 }, dim_location[LocationId] ),
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    TREATAS ( { DATE ( 2026, 9, 15 ) }, dim_date[Date] ),
    "Mean of hours present", AVERAGE ( fact_measurement[Value] ),
    "Hours present", COUNTROWS ( fact_measurement )
)""",
    4: """
EVALUATE
SUMMARIZECOLUMNS (
    dim_location[LocationName],
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    TREATAS ( { DATE ( 2026, 9, 15 ) }, dim_date[Date] ),
    "Mean of hours present", AVERAGE ( fact_measurement[Value] ),
    "Hours present", COUNTROWS ( fact_measurement )
)
ORDER BY [Mean of hours present] DESC""",
    5: """
EVALUATE
SUMMARIZECOLUMNS (
    dim_date[YearMonth],
    TREATAS ( { 2993 }, dim_location[LocationId] ),
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    TREATAS ( { "2026-09" }, dim_date[YearMonth] ),
    "PM2.5 readings", COUNTROWS ( fact_measurement )
)""",
    6: """
EVALUATE
FILTER (
    ADDCOLUMNS (
        VALUES ( dim_location[LocationName] ),
        "PM2.5 readings",
            CALCULATE (
                COUNTROWS ( fact_measurement ),
                dim_parameter[ParameterName] = "pm25",
                dim_date[Date] = DATE ( 2026, 9, 12 )
            )
    ),
    ISBLANK ( [PM2.5 readings] )
)""",
    7: """
DEFINE
    VAR Berlin = TREATAS ( { 2993 }, dim_location[LocationId] )
    VAR Pm25 = TREATAS ( { "pm25" }, dim_parameter[ParameterName] )
    VAR DaysWithData = CALCULATETABLE ( SUMMARIZE ( fact_measurement, dim_date[Date] ), Berlin, Pm25 )
EVALUATE
ADDCOLUMNS (
    TOPN ( 30, DaysWithData, dim_date[Date], DESC ),
    "Mean of hours present", CALCULATE ( AVERAGE ( fact_measurement[Value] ), Berlin, Pm25 ),
    "Hours present", CALCULATE ( COUNTROWS ( fact_measurement ), Berlin, Pm25 )
)
ORDER BY dim_date[Date]""",
    8: """
EVALUATE
SUMMARIZECOLUMNS (
    dim_date[Date],
    dim_location[LocationName],
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    DATESBETWEEN ( dim_date[Date], DATE ( 2026, 9, 1 ), DATE ( 2026, 9, 30 ) ),
    "Mean of hours present", AVERAGE ( fact_measurement[Value] ),
    "Hours present", COUNTROWS ( fact_measurement )
)
ORDER BY dim_date[Date], dim_location[LocationName]""",
    9: """
EVALUATE
CALCULATETABLE (
    SUMMARIZE ( dim_location, dim_location[LocationName], dim_location[ProviderName], dim_location[OwnerName] ),
    dim_location[LocationId] = 2993
)""",
    10: """
DEFINE
    VAR Guideline =
        CALCULATE (
            MAX ( ref_guideline[GuidelineValue] ),
            ref_guideline[ParameterName] = "pm25",
            ref_guideline[AveragingPeriod] = "24-hour"
        )
EVALUATE
SUMMARIZECOLUMNS (
    dim_date[Date],
    TREATAS ( { 2993 }, dim_location[LocationId] ),
    TREATAS ( { "pm25" }, dim_parameter[ParameterName] ),
    "Mean of hours present", AVERAGE ( fact_measurement[Value] ),
    "Hours present", COUNTROWS ( fact_measurement ),
    "WHO 24-hour guideline", IF ( NOT ISBLANK ( AVERAGE ( fact_measurement[Value] ) ), Guideline )
)
ORDER BY dim_date[Date]""",
}

for number, query in REFERENCE_QUERIES.items():
    print(f"Question {number}")
    display(fabric.evaluate_dax(MODEL_NAME, query))